In [1]:
import pyspark
from pyspark.sql import SparkSession

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA, Imputer
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.sql.functions import mean, col, expr
import numpy as np
import time

In [2]:
spark = SparkSession.builder.appName("rka7 - Dataset Reduction") \
    .config("SPARK_LOCAL_IP", "192.168.1.2") \
    .master("spark://192.168.1.2:7077") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.maxResultSize", "3g") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "25g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "16") \
    .config("spark.shuffle.partitions", "180") \
    .config("spark.kryoserializer.buffer.max", "256m") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.sql.execution.arrow.enabled", "true") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/06/09 17:07:15 WARN Utils: Your hostname, ubuntu-virtual-machine resolves to a loopback address: 127.0.1.1; using 192.168.1.107 instead (on interface ens33)
24/06/09 17:07:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/06/09 17:07:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#spark.sparkContext.stop()

In [4]:
parquet_files = ["hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2021-12-12 - 2021-12-19/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2021-12-19 - 2021-12-26/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2021-12-26 - 2022-01-02/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-01-02 - 2022-01-09/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-01-09 - 2022-01-16/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-01-16 - 2022-01-23/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-02-06 - 2022-02-13/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet",
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekData22/parquet/2022-02-13 - 2022-02-20/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet"]

In [5]:
#Read the parquet files
df = spark.read.parquet(*parquet_files, inferSchema=True)

24/06/09 17:07:28 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/06/09 17:07:28 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/06/09 17:07:28 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


In [6]:
#Get unique label counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [7]:
start_time = time.time()

#Drop labels and get remaining counts
labels_to_drop = ['Defense Evasion', 
                  'Exfiltration',
                  'Initial Access',
                  'Lateral Movement', 
                  'Persistence',
                  'Privilege Escalation', 
                  'Resource Development', 
                  'Credential Access',
                  'Discovery']
                  #'Reconnaissance']

df = df.filter(~col("label_tactic").isin(labels_to_drop))

#Get unique label counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

label_counts = df.groupBy("label_tactic").count().collect()
total_count = df.count()
print(total_count)

# Calculate sampling fractions/proportions for each stratum/category (Doing Stratified Sampling
fractions = {row['label_tactic']: row['count'] / total_count for row in label_counts}

print(fractions)

# Sample size (85% reduction of dataframe)
sample_val = total_count * 0.15
sample_size = round(sample_val)


# Calculate the fraction to sample for the required sample size
sample_fractions = {label_tactic: (fraction * sample_size / total_count) for label_tactic, fraction in fractions.items()}


# Perform stratified sampling
stratified_sample = df.sampleBy("label_tactic", fractions=sample_fractions, seed=42)

# Replace dataframe w/ reduced dataset
df = stratified_sample
df.count()

label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

+--------------+-------+
|  label_tactic|  count|
+--------------+-------+
|Reconnaissance|9278722|
|          none|9281599|
+--------------+-------+



18560321
{'Reconnaissance': 0.49992249595252153, 'none': 0.5000775040474785}


+--------------+------+
|  label_tactic| count|
+--------------+------+
|Reconnaissance|696198|
|          none|694865|
+--------------+------+

Execution time: 9.904124975204468 seconds


In [8]:
#Drop uid feature
df = df.drop('uid')

In [9]:
start_time = time.time()

#Columns to index
columns_to_index = ['service', 
                    'conn_state', 
                    'history', 
                    'proto', 
                    'dest_ip_zeek', 
                    'community_id', 
                    'src_ip_zeek',
                    'datetime',
                    'local_resp',
                    'local_orig',
                    'label_tactic']

#Cast datetime, local_resp, local_orig to String
df = df.withColumn("datetime", col("datetime").cast("string"))
df = df.withColumn("local_resp", col("local_resp").cast("string"))
df = df.withColumn("local_orig", col("local_orig").cast("string"))

#Impute null values with empty string
for column in columns_to_index:
    df = df.fillna('', subset=[column])

In [10]:
#Split the into training and test sets
start_time = time.time()
train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.030067920684814453 seconds


In [11]:
#StringIndexer
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").setHandleInvalid("keep") for column in columns_to_index]

#Chain indexers together
pipeline = Pipeline(stages=indexers).fit(train_data)

#Fit and transform the data
train_data_indexed = pipeline.transform(train_data)
test_data_indexed = pipeline.transform(test_data)

#Drop original columns
train_data_indexed = train_data_indexed.drop(*columns_to_index)
train_data_indexed = train_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")
test_data_indexed = test_data_indexed.drop(*columns_to_index)
test_data_indexed = test_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")

#print("train_indexed columns: ", train_data_indexed.columns)
#print("test_indexed columns: ", test_data_indexed.columns)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 58.830283641815186 seconds


In [12]:
start_time = time.time()

#List of numeric column names
numeric_columns = ['resp_pkts', 
                   'orig_ip_bytes', 
                   'missed_bytes', 
                   'duration', 
                   'orig_pkts',
                   'resp_ip_bytes', 
                   'dest_port_zeek', 
                   'orig_bytes', 
                   'resp_bytes',
                   'src_port_zeek', 
                   'ts']

#Create Imputer
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

#Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data_indexed)

#Apply the Imputer to the training data
train_data_imputed = imputer_model.transform(train_data_indexed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Apply the Imputer to the test data
start_time = time.time()
test_data_imputed = imputer_model.transform(test_data_indexed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

train_data_imputed = train_data_imputed.drop(*numeric_columns)
test_data_imputed = test_data_imputed.drop(*numeric_columns)

#print("\nTrain data imputed: ", train_data_imputed.columns)
#print("\n")
#print("Test data imputed: ", test_data_imputed.columns)

Execution time: 5.70185661315918 seconds
Execution time: 0.0196077823638916 seconds


In [13]:
start_time = time.time()

#Create VectorAssembler
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed") or column.endswith("_indexed")]
#print("Columns to assemble: ", columns_to_assemble)

assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

#Transform the training data
train_data_assembled = assembler.transform(train_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Transform the test data
start_time = time.time()
test_data_assembled = assembler.transform(test_data_imputed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Select the features and label columns
train_data_assembled = train_data_assembled.select("features", "label_tactic")
test_data_assembled = test_data_assembled.select("features", "label_tactic")

#print("Train_data_assembled columns: ", train_data_assembled.columns)
#print("Test_data_assembled columns: ", test_data_assembled.columns)

Execution time: 1.2894024848937988 seconds
Execution time: 0.6300909519195557 seconds


In [14]:
from pyspark.ml.feature import StandardScaler

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic")

24/06/09 17:09:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/06/09 17:09:15 WARN DAGScheduler: Broadcasting large task binary with size 30.2 MiB
24/06/09 17:09:23 WARN DAGScheduler: Broadcasting large task binary with size 30.2 MiB


In [15]:
# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic")

In [16]:

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd

start_time = time.time()

# Convert Spark DataFrame to Pandas DataFrame
train_pd = train_data_normalized.toPandas()
test_pd = test_data_normalized.toPandas()

# Extract features and labels from Pandas DataFrames
X_train = train_pd['features_normalized'].values.tolist()
y_train = train_pd['label_tactic'].values.tolist()
X_test = test_pd['features_normalized'].values.tolist()
y_test = test_pd['label_tactic'].values.tolist()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Define the number of components for LDA
n_components = 1 

start_time = time.time()

# Perform Linear Discriminant Analysis in scikit-learn with the specified number of components
lda = LinearDiscriminantAnalysis(n_components=n_components)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()

# Convert the transformed arrays back to Pandas DataFrames
train_pd_lda = pd.DataFrame(X_train_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])
test_pd_lda = pd.DataFrame(X_test_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])

# Combine the transformed features with the labels
train_pd_lda['label_tactic'] = y_train
test_pd_lda['label_tactic'] = y_test

# Convert Pandas DataFrames back to Spark DataFrames
train_lda = spark.createDataFrame(train_pd_lda)
test_lda = spark.createDataFrame(test_pd_lda)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Show the adjusted data
train_lda.show()
test_lda.show()

24/06/09 17:09:28 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
24/06/09 17:09:56 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB


Execution time: 45.90542984008789 seconds
Execution time: 13.533947467803955 seconds
Execution time: 20.77009677886963 seconds


+------------------+------------+
|     lda_feature_1|label_tactic|
+------------------+------------+
|  4.18006923153763|         1.0|
| 4.176654124929056|         1.0|
| 4.166718580139915|         1.0|
|4.1349221111009085|         1.0|
|4.4032478658699175|         1.0|
| 4.400275617413761|         1.0|
| 4.381396333285377|         1.0|
| 4.377954095034082|         1.0|
| 4.362148689827755|         1.0|
|  4.35264341990006|         1.0|
| 3.858405202975695|         1.0|
|3.8038499170426534|         1.0|
|3.7972704419313397|         1.0|
| 3.089123008690401|         1.0|
| 3.089123008690401|         1.0|
| 5.572470708928543|         1.0|
| 5.568864059495084|         1.0|
| 5.568325887988683|         1.0|
| 5.566527746733379|         1.0|
|  5.56641800089173|         1.0|
+------------------+------------+
only showing top 20 rows

+------------------+------------+
|     lda_feature_1|label_tactic|
+------------------+------------+
| 6.508879644584929|         1.0|
| 6.393549930754938|  

In [17]:
start_time = time.time()

# List of columns to assemble
columns_to_assemble = [f'lda_feature_{i+1}' for i in range(n_components)]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform train_lda
train_lda_assembled = assembler.transform(train_lda)

# Transform test_lda
test_lda_assembled = assembler.transform(test_lda)

# Select only the assembled features and label column for both datasets
train_lda_assembled = train_lda_assembled.select("features", "label_tactic")
test_lda_assembled = test_lda_assembled.select("features", "label_tactic")

# Show the schema of the assembled train_lda DataFrame
train_lda_assembled.printSchema()

# Show the schema of the assembled test_lda DataFrame
test_lda_assembled.printSchema()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

root
 |-- features: vector (nullable = true)
 |-- label_tactic: double (nullable = true)

root
 |-- features: vector (nullable = true)
 |-- label_tactic: double (nullable = true)

Execution time: 0.27503108978271484 seconds


In [18]:
#Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)
ovr = OneVsRest(classifier=svm, labelCol="label_tactic")
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.09868168830871582 seconds


In [19]:
#Fit the model
start_time = time.time()
svm_model = ovr.fit(train_lda_assembled)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/06/09 17:10:49 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


Execution time: 23.27558708190918 seconds


In [20]:
#Make predictions
start_time = time.time()

predictions = svm_model.transform(test_lda_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/06/09 17:11:11 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


Execution time: 0.4017307758331299 seconds


In [21]:
# Evaluate the model
# Calculate accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)

# Calculate precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Calculate recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Calculate F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

#Calculate FPR
evaluator_fprL = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="falsePositiveRateByLabel")
fprL_score = evaluator_fprL.evaluate(predictions)

#Calculate Weighted FPR
evaluator_fpr = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedFalsePositiveRate")
fpr_score = evaluator_fpr.evaluate(predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)
print("FPR by Label:", fprL_score)
print("Weighted FPR:", fpr_score)

Accuracy: 0.9995908121780038
Precision: 0.9995909905274407
Recall: 0.9995908121780038
F1-Score: 0.9995908119196848
FPR by Label: 0.000709587097021652
Weighted FPR: 0.0004102741609706245


In [22]:
start_time = time.time()

#Extract predictions and labels
predictions_and_labels = predictions.select("prediction", "label_tactic")

#Calculate false positives and true negatives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic== 0)).count()
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic == 0)).count()

#Calculate FPR
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

False Positive Rate: 0.00010987488594509121
Execution time: 1.6057415008544922 seconds


In [23]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import Row

# Convert DataFrame to RDD of tuples (prediction, label)
prediction_and_labels = predictions.select("prediction", "label_tactic") \
    .rdd.map(lambda row: (float(row['prediction']), float(row['label_tactic'])))


# Instantiate BinaryClassificationMetrics
metrics = BinaryClassificationMetrics(prediction_and_labels)

# Compute AUROC
auROC = metrics.areaUnderROC

# Print AUROC
print("Area under ROC = ", auROC)

/home/ubuntu/.local/lib/python3.10/site-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Area under ROC =  0.9995902690085167


In [24]:
spark.sparkContext.stop()